# Nassau Candy Distributor — Exploratory Data Analysis

**Author:** Nassau Candy Analytics Team  
**Dataset:** Nassau Candy Distributor order records  
**Objective:** Understand shipping lead times, factory performance, and product-level logistics patterns to inform the factory reallocation recommendation system.

---

## Table of Contents
1. [Data Loading & Overview](#1)
2. [Lead Time Distribution](#2)
3. [Factory Performance Analysis](#3)
4. [Product-Level Insights](#4)
5. [Shipping Distance vs Lead Time](#5)
6. [Regional Analysis](#6)
7. [Profitability by Division](#7)
8. [Key Findings & Recommendations](#8)

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded successfully.')

## 1. Data Loading & Overview <a id='1'></a>

We load the preprocessed dataset which includes computed shipping distances, lead times, factory assignments, and profit margins derived from the raw order data.

In [ ]:
df = pd.read_csv('../data/nassau_processed.csv', parse_dates=['Order Date', 'Ship Date'])
print(f'Shape: {df.shape}')
print(f'Date range: {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
df.head()

In [ ]:
print('=== Dataset Summary ===')
print(f'Total orders         : {len(df):,}')
print(f'Unique products      : {df["Product Name"].nunique()}')
print(f'Unique factories     : {df["Factory"].nunique()}')
print(f'Unique states        : {df["State/Province"].nunique()}')
print(f'Unique ship modes    : {df["Ship Mode"].nunique()}')
print(f'Total sales ($)      : {df["Sales"].sum():,.0f}')
print(f'Total gross profit ($): {df["Gross Profit"].sum():,.0f}')
print(f'Avg lead time (days) : {df["Lead Time"].mean():.2f}')
print(f'Avg distance (miles) : {df["Distance Miles"].mean():.0f}')

In [ ]:
# Missing values check
missing = df.isnull().sum()
missing[missing > 0]

## 2. Lead Time Distribution <a id='2'></a>

Lead time (days from order to shipment) is the primary target variable for our predictive models. Understanding its distribution helps identify structural patterns and outliers.

In [ ]:
fig = px.histogram(
    df, x='Lead Time', nbins=30,
    title='Distribution of Shipping Lead Time',
    labels={'Lead Time': 'Lead Time (days)'},
    color_discrete_sequence=['#1D9E75'],
)
fig.add_vline(x=df['Lead Time'].mean(), line_dash='dash', line_color='red',
              annotation_text=f'Mean: {df["Lead Time"].mean():.1f}d')
fig.add_vline(x=df['Lead Time'].median(), line_dash='dot', line_color='blue',
              annotation_text=f'Median: {df["Lead Time"].median():.1f}d')
fig.show()

In [ ]:
print('Lead Time Statistics:')
print(df['Lead Time'].describe().round(2))

## 3. Factory Performance Analysis <a id='3'></a>

We compare average lead time, shipping distance, and order volume across all five factories to identify underperforming routes.

In [ ]:
factory_stats = df.groupby('Factory').agg(
    Orders=('Lead Time', 'count'),
    Avg_Lead_Time=('Lead Time', 'mean'),
    Median_Lead_Time=('Lead Time', 'median'),
    Std_Lead_Time=('Lead Time', 'std'),
    Avg_Distance=('Distance Miles', 'mean'),
    Avg_Sales=('Sales', 'mean'),
    Avg_Profit_Margin=('Profit Margin', 'mean'),
).round(2).reset_index().sort_values('Avg_Lead_Time')

factory_stats

In [ ]:
fig = px.box(
    df, x='Factory', y='Lead Time',
    color='Factory',
    title='Lead Time Distribution by Factory',
    points='outliers',
    labels={'Lead Time': 'Lead Time (days)'},
)
fig.update_layout(showlegend=False)
fig.show()

## 4. Product-Level Insights <a id='4'></a>

Products vary significantly in their shipping profiles depending on the assigned factory and destination regions.

In [ ]:
product_stats = df.groupby(['Product Name', 'Factory']).agg(
    Orders=('Lead Time', 'count'),
    Avg_Lead_Time=('Lead Time', 'mean'),
    Avg_Distance=('Distance Miles', 'mean'),
    Avg_Margin=('Profit Margin', 'mean'),
).round(2).reset_index().sort_values('Avg_Lead_Time', ascending=False)

product_stats.head(15)

In [ ]:
fig = px.bar(
    product_stats,
    x='Avg_Lead_Time', y='Product Name',
    color='Factory',
    orientation='h',
    title='Average Lead Time by Product (coloured by Factory)',
    labels={'Avg_Lead_Time': 'Avg Lead Time (days)', 'Product Name': ''},
    height=500,
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 5. Shipping Distance vs Lead Time <a id='5'></a>

We examine whether shipping distance is a strong predictor of lead time — a key assumption for the optimization model.

In [ ]:
corr = df[['Lead Time', 'Distance Miles', 'Units', 'Cost', 'Sales', 'Gross Profit']].corr()
print('Correlation with Lead Time:')
print(corr['Lead Time'].sort_values(ascending=False))

In [ ]:
fig = px.scatter(
    df.sample(min(3000, len(df)), random_state=42),
    x='Distance Miles', y='Lead Time',
    color='Factory', opacity=0.5,
    title='Shipping Distance vs Lead Time by Factory',
    trendline='ols',
    labels={'Distance Miles': 'Distance (miles)', 'Lead Time': 'Lead Time (days)'},
)
fig.show()

## 6. Regional Analysis <a id='6'></a>

Lead times vary by destination region — some regions consistently receive slower shipments regardless of product.

In [ ]:
regional = df.groupby('Region').agg(
    Orders=('Lead Time', 'count'),
    Avg_Lead_Time=('Lead Time', 'mean'),
    Avg_Distance=('Distance Miles', 'mean'),
).round(2).reset_index().sort_values('Avg_Lead_Time', ascending=False)

print(regional.to_string(index=False))

In [ ]:
fig = px.bar(
    regional,
    x='Region', y='Avg_Lead_Time',
    color='Avg_Distance',
    color_continuous_scale='RdYlGn_r',
    title='Average Lead Time by Region (colour = avg distance)',
    labels={'Avg_Lead_Time': 'Avg Lead Time (days)', 'Avg_Distance': 'Avg Distance (mi)'},
)
fig.show()

In [ ]:
# Ship mode breakdown
shipmode_stats = df.groupby('Ship Mode').agg(
    Orders=('Lead Time', 'count'),
    Avg_Lead_Time=('Lead Time', 'mean'),
).round(2).reset_index().sort_values('Avg_Lead_Time')

fig = px.bar(
    shipmode_stats, x='Ship Mode', y='Avg_Lead_Time',
    color='Ship Mode',
    title='Average Lead Time by Ship Mode',
    labels={'Avg_Lead_Time': 'Avg Lead Time (days)'},
)
fig.update_layout(showlegend=False)
fig.show()

## 7. Profitability by Division <a id='7'></a>

We assess profit margins across product divisions to understand which segments are most sensitive to logistics cost changes.

In [ ]:
division_stats = df.groupby('Division').agg(
    Orders=('Gross Profit', 'count'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Gross Profit', 'sum'),
    Avg_Margin=('Profit Margin', 'mean'),
).round(2).reset_index()

division_stats['Margin %'] = (division_stats['Avg_Margin'] * 100).round(1)
print(division_stats.to_string(index=False))

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['Total Sales by Division', 'Avg Profit Margin (%)'])

fig.add_trace(go.Bar(
    x=division_stats['Division'],
    y=division_stats['Total_Sales'],
    marker_color=['#1D9E75', '#378ADD', '#E85D24'],
    name='Total Sales'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=division_stats['Division'],
    y=division_stats['Margin %'],
    marker_color=['#1D9E75', '#378ADD', '#E85D24'],
    name='Avg Margin %'
), row=1, col=2)

fig.update_layout(title='Division-Level Financial Profile', showlegend=False, height=380)
fig.show()

## 8. Key Findings & Recommendations <a id='8'></a>

### Summary of Findings

| Finding | Detail |
|---|---|
| Lead time range | Varies from 0 to ~14 days depending on factory-region pairing |
| Distance-lead time correlation | Moderate positive correlation; distance is a significant predictor |
| Factory performance gap | Some factories show consistently higher lead times for certain regions |
| Ship mode impact | First Class significantly outperforms Standard Class on lead time |
| Profitability risk | Chocolate division has highest sales but margin sensitivity to distance changes |

### Strategic Recommendations

1. **Run the optimizer** (Pages 1–3 of the dashboard) to identify the highest-impact product reassignments.
2. **Prioritise Sugar Shack products** shipping to the West — the geographic mismatch is largest.
3. **Monitor Chocolate division margins** closely when simulating factory swaps — any distance increase erodes already thin margins.
4. **Validate recommendations** against actual freight costs before executing reassignments.